# Notebook 02: Hệ thống Câu hỏi Nghiên cứu & Tổng quan Phân tích Chuyên sâu (Master Hub)
**Đề tài:** Phân tích Sự cố Bất lợi Thuốc Thú y (OpenFDA Animal & Veterinary Adverse Events) giai đoạn 2025 và Đối chiếu Xu hướng 2026 (Q1)

**Môn học:** CSC17104 – Lập trình cho Khoa học Dữ liệu  
**Nhóm thực hiện:** Global Ban (Đinh Công Khang - M1, Trần Ngô Uyên Nhi - M2, Huỳnh Tuấn Kiệt - M3)  

> **Kiến trúc Phân rã Notebook theo Đề tài (Rubric Section 4.3 — `split per question if large`):**  
> Để tối ưu hóa quy trình làm việc nhóm, chống tràn bộ nhớ và bảo đảm tính độc lập chuyên sâu, toàn bộ phân tích chi tiết cho 5 câu hỏi Non-ML được phân rã thành **3 Notebook chuyên biệt chính thức** tương ứng với 3 nhánh phát triển của từng thành viên:
> 1. **`02a_analysis_khang.ipynb`**: **Câu hỏi Q1** (VeDDRA & Chuỗi thời gian 5 quý — Owner: Khang — M1)
> 2. **`02b_analysis_nhi.ipynb`**: **Câu hỏi Q2** (Sinh trắc học & Biến cố nghiêm trọng) & **Câu hỏi Q5** (Tỷ lệ ảnh hưởng ở đàn gia súc — Attack Rate — Owner: Nhi — M2)
> 3. **`02c_analysis_kiet.ipynb`**: **Câu hỏi Q3** (Tỷ suất ROR Hoạt chất x Triệu chứng) & **Câu hỏi Q4** (Sử dụng thuốc sai nhãn & Kết cục y khoa — Owner: Kiệt — M3)
>
> File Master này (`02_analysis.ipynb`) đóng vai trò là **Trung tâm Điều phối (Master Hub)**: Trình bày Mục 3.3 (Hệ thống câu hỏi & Ma trận độ sâu D1–D6) và Tổng hợp Kết quả then chốt (Executive Summary) của 5 câu hỏi phân tích.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.data_loader import load_processed
from src.visualization import set_project_plot_style
set_project_plot_style()

# Tải 4 bảng dữ liệu đã qua tiền xử lý chuẩn hóa từ data/processed/
data = load_processed(PROJECT_ROOT / 'data' / 'processed')
reports = data['reports']
drugs = data['drugs']
reactions = data['reactions']
outcomes = data['outcomes']
print(f"[Data Loaded] Reports: {reports.shape}, Drugs: {drugs.shape}, Reactions: {reactions.shape}, Outcomes: {outcomes.shape}")

---
## 1. Tổng quan Hệ thống Câu hỏi Nghiên cứu & Ma trận Độ sâu (Mục 3.3)

Theo yêu cầu của môn học, nhóm 3 thành viên xây dựng **$2 \times 3 = 6$ câu hỏi** (5 câu hỏi phân tích thống kê/khám phá chuyên sâu tại các notebook `02a`, `02b`, `02c` và 1 câu hỏi Học máy tại `03_ml_question.ipynb`). Toàn bộ bộ câu hỏi bao phủ trọn vẹn **6/6 tiêu chí độ sâu (D1–D6)** và mỗi câu hỏi Non-ML đáp ứng ít nhất 3 tiêu chí:

| Mã | Chủ sở hữu | Câu hỏi nghiên cứu | Tiêu chí độ sâu đáp ứng | Notebook thực thi |
|---|---|---|---|---|
| **Q1** | **M1 — Khang** | Những phản ứng bất lợi (VeDDRA terms) nào chiếm tỷ trọng cao nhất ở Chó, Mèo và các loài khác trong năm 2025, và xu hướng biến động theo quý (kèm dấu hiệu mùa vụ trong phạm vi 1 năm) của chúng diễn biến ra sao khi bước sang nửa đầu năm 2026 (Q1)? | **D1** (Combining tables), **D2** (Reshaping & time series), **D5** (Quantifying uncertainty) | [`02a_analysis_khang.ipynb`](02a_analysis_khang.ipynb) |
| **Q2** | **M2 — Nhi** | Những đặc điểm sinh học nào (giai đoạn sống, trọng lượng cơ thể, giới tính) có liên quan đến các biến cố bất lợi nghiêm trọng, sau khi đã kiểm soát yếu tố loài và số lượng thuốc sử dụng? | **D3** (Derived variables), **D4** (Controlling for confounder), **D5** (CI/Hypothesis test), **D6** (Robustness check) | [`02b_analysis_nhi.ipynb`](02b_analysis_nhi.ipynb) |
| **Q3** | **M3 — Kiệt** | Các hoạt chất (`active_ingredients`) và đường dùng (`route`) nào có tỷ suất đồng xuất hiện (Reporting Odds Ratio) đặc trưng nhất với từng nhóm phản ứng cụ thể, và cấu trúc này khác biệt ra sao giữa **Safety Issue** và **Lack Of Expected Effectiveness** trong năm 2025? | **D1** (Multi-table merge), **D2** (Pivot/melt matrix), **D4** (Subgroup control), **D5** (Odds Ratio & 95% CI) | [`02c_analysis_kiet.ipynb`](02c_analysis_kiet.ipynb) |
| **Q4** | **M3 — Kiệt** | Việc dùng thuốc sai nhãn (`off-label`) kết hợp với đa trị liệu (`polypharmacy`) trong năm 2025 có liên hệ như thế nào với mức độ nghiêm trọng (`serious_ae`) và kết cục (`outcome`) sau khi kiểm soát biến nhiễu là **loài** và **đối tượng báo cáo (`primary_reporter`)**? | **D1** (Combining tables), **D3** (Derived indicators), **D4** (Controlling for confounder), **D6** (Robustness check) | [`02c_analysis_kiet.ipynb`](02c_analysis_kiet.ipynb) |
| **Q5** | **M2 — Nhi** | Những yếu tố nào có liên quan đến tỷ lệ động vật bị ảnh hưởng trong tổng số động vật được điều trị ở mỗi đàn? | **D1** (Multi-table join for herd data), **D3** (Derived attack rate), **D4** (Subgroup / species control), **D5** (CI & regression), **D6** (Robustness: exclude attack_rate > 1 vs. keep) | [`02b_analysis_nhi.ipynb`](02b_analysis_nhi.ipynb) |
| **Q6 (ML)** | **M1 — Khang** | Huấn luyện mô hình trên dữ liệu 2025 (Q1–Q4) để dự đoán khả năng một báo cáo trong năm 2026 (Q1) là sự cố nghiêm trọng (`serious_ae`) dựa trên thông tin ban đầu lúc tiếp nhận. | **ML Checklist 1–6** (Time-based split, Pipeline, Leakage-free, Tuning, Error Analysis, Permutation Importance) | [`03_ml_question.ipynb`](03_ml_question.ipynb) |

---
## 2. Câu hỏi Q1: Phản ứng VeDDRA hàng đầu & Xu hướng Biến động theo Quý
- **Chủ sở hữu:** Đinh Công Khang (M1) — Nhánh Git: `feature/khang`
- **Notebook chi tiết (Code, Biểu đồ & Kiểm định):** 👉 [`02a_analysis_khang.ipynb`](02a_analysis_khang.ipynb)
- **Tóm tắt Phát hiện & Con số then chốt (Executive Summary):**
  - *Top phản ứng VeDDRA theo loài:* (Sẽ được trích xuất từ `02a_analysis_khang.ipynb` khi hoàn tất phân tích)
  - *Dấu hiệu mùa vụ & Biến động 2025 $\rightarrow$ 2026 Q1:* (Sẽ được trích xuất từ `02a_analysis_khang.ipynb`)
  - *Giá trị p-value kiểm định $\chi^2$ & Bootstrap 95% CI:* (Sẽ được trích xuất từ `02a_analysis_khang.ipynb`)

---
## 3. Câu hỏi Q2: Đặc điểm Sinh học & Nguy cơ Biến cố Nghiêm trọng
- **Chủ sở hữu:** Trần Ngô Uyên Nhi (M2) — Nhánh Git: `feature/nhi`
- **Notebook chi tiết (Code, Biểu đồ & Kiểm định):** 👉 [`02b_analysis_nhi.ipynb`](02b_analysis_nhi.ipynb)
- **Tóm tắt Phát hiện & Con số then chốt (Executive Summary):**
  - *Ảnh hưởng của giai đoạn sống & Cân nặng:* (Sẽ được trích xuất từ `02b_analysis_nhi.ipynb` khi hoàn tất phân tích)
  - *Tỷ suất chênh điều chỉnh (Adjusted Odds Ratio) & 95% CI:* (Sẽ được trích xuất từ `02b_analysis_nhi.ipynb`)
  - *Kết quả kiểm tra độ vững D6:* (Sẽ được trích xuất từ `02b_analysis_nhi.ipynb`)

---
## 4. Câu hỏi Q3: Tỷ suất Đồng xuất hiện (ROR) Hoạt chất x Triệu chứng (Safety vs Efficacy)
- **Chủ sở hữu:** Huỳnh Tuấn Kiệt (M3) — Nhánh Git: `feature/kiet`
- **Notebook chi tiết (Code, Biểu đồ & Kiểm định):** 👉 [`02c_analysis_kiet.ipynb`](02c_analysis_kiet.ipynb)
- **Tóm tắt Phát hiện & Con số then chốt (Executive Summary):**
  - *Cặp Hoạt chất — Triệu chứng có tín hiệu an toàn mạnh nhất (ROR > 2):* (Sẽ được trích xuất từ `02c_analysis_kiet.ipynb` khi hoàn tất phân tích)
  - *Khác biệt cấu trúc giữa Safety Issue vs Lack of Efficacy:* (Sẽ được trích xuất từ `02c_analysis_kiet.ipynb`)
  - *Khoảng tin cậy 95% CI và kết quả phân tầng loài:* (Sẽ được trích xuất từ `02c_analysis_kiet.ipynb`)

---
## 5. Câu hỏi Q4: Sử dụng Thuốc Sai Nhãn (Off-label), Đa Trị Liệu & Kết cục Y khoa
- **Chủ sở hữu:** Huỳnh Tuấn Kiệt (M3) — Nhánh Git: `feature/kiet`
- **Notebook chi tiết (Code, Biểu đồ & Kiểm định):** 👉 [`02c_analysis_kiet.ipynb`](02c_analysis_kiet.ipynb)
- **Tóm tắt Phát hiện & Con số then chốt (Executive Summary):**
  - *Tỷ lệ gia tăng nguy cơ tử vong khi dùng off-label:* (Sẽ được trích xuất từ `02c_analysis_kiet.ipynb` khi hoàn tất phân tích)
  - *Tác động cộng hưởng giữa Off-label và Đa trị liệu (Polypharmacy):* (Sẽ được trích xuất từ `02c_analysis_kiet.ipynb`)
  - *Kiểm soát biến nhiễu người báo cáo (`primary_reporter`) và kết quả D6:* (Sẽ được trích xuất từ `02c_analysis_kiet.ipynb`)

---
## 6. Câu hỏi Q5: Các yếu tố liên quan đến Tỷ lệ Động vật Bị ảnh hưởng ở Đàn (Herd Attack Rate)
- **Chủ sở hữu:** Trần Ngô Uyên Nhi (M2) — Nhánh Git: `feature/nhi`
- **Notebook chi tiết (Code, Biểu đồ & Kiểm định):** 👉 [`02b_analysis_nhi.ipynb`](02b_analysis_nhi.ipynb)
- **Tóm tắt Phát hiện & Con số then chốt (Executive Summary):**
  - *Mức độ ảnh hưởng trung vị theo loài đàn và đường dùng thuốc:* (Sẽ được trích xuất từ `02b_analysis_nhi.ipynb` khi hoàn tất phân tích)
  - *Giá trị kiểm định Kruskal-Wallis & Bootstrap 95% CI:* (Sẽ được trích xuất từ `02b_analysis_nhi.ipynb`)
  - *Kiểm tra độ vững D6 khi xử lý các ca attack_rate > 1:* (Sẽ được trích xuất từ `02b_analysis_nhi.ipynb`)